# Bronze Layer
------------------
- Data for all international games from 1872 to 2026
- To read in source kaggle api data
- To make sure data working well

# Import necessary libraries
---------------------


In [0]:
%pip install kagglehub[pandas-datasets]

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import requests
import kagglehub
from kagglehub import KaggleDatasetAdapter
from utill_funcs import get_winner
import plotly.express as px

# Parameters
---------------

In [0]:
catalog = 'wc_2026_predions'
schema = 'bronze'
bronze_table_name = 'source'            

# Read in the API
-------------------
- A kaggle api using the data International football results from 1872 to 2026 at https://www.kaggle.com/datasets/martj42/international-football-results-from-1872-to-2017?resource=download

In [0]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas

# Set the path to the file you'd like to load
file_path = "results.csv"
# phone testing
# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "martj42/international-football-results-from-1872-to-2017",
  file_path,
)

print("First 5 records:", df.head())

In [0]:
display(df[df['date'].astype(str).str.contains('2026')])

# Get Data Size
-----------------

In [0]:
print(f'The df shape : {df.shape}')
print(f'The df columns : {df.columns}')
print(f'The df dtypes : {df.dtypes}')


### Visual EDA
-------------------
- Lets see patterns with visualizations

In [0]:
display(df)

# Add column to define the winner
------------------------------------
- Determine the winner, loser and draw results

In [0]:
df['winner'] = df.apply(get_winner, axis=1)
display(df)

In [0]:
### EDA with Visualizations ofn results
# Plot the distribution of tournament types
tournament_counts = df['tournament'].value_counts()
fig = px.bar(tournament_counts, x=tournament_counts.index, y=tournament_counts.values, labels={'x': 'Tournament', 'y': 'Count'})
fig.update_layout(title='Distribution of Tournament Types')
fig.show()
# Plot the distribution of home and away scores
scores = df[['home_score', 'away_score']]
fig = px.histogram(scores, x=scores.columns, nbins=20, labels={'x': 'Score', 'y': 'Count'})
fig.update_layout(title='Distribution of Home and Away Scores')
fig.show()

#Write Bronze layer
-------------------------
- If catalog and schema not exist create else overwrite this layer.

In [0]:
spark.sql("CREATE CATALOG IF NOT EXISTS wc_2026_predions")
spark.sql("CREATE SCHEMA IF NOT EXISTS wc_2026_predions.bronze")

In [0]:
spark_df = spark.createDataFrame(df)
spark_df.write.format("delta").mode("overwrite").saveAsTable(f'{catalog}.{schema}.{bronze_table_name}')